In [1]:
import numpy as np
import pandas as pd 
import os
os.chdir('..')  # Go up to project root
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
for dirname, _, filenames in os.walk('./kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./kaggle/input/broad-peak.csv
./kaggle/input/lhotse.csv
./kaggle/input/kangchenjunga.csv
./kaggle/input/.DS_Store
./kaggle/input/cho-oyu.csv
./kaggle/input/everest_20-25.csv
./kaggle/input/manaslu.csv
./kaggle/input/dhaulagiri-I.csv
./kaggle/input/everest.csv
./kaggle/input/k2.csv
./kaggle/input/nanga-parbat.csv
./kaggle/input/gasherbrum-II.csv
./kaggle/input/shishapangma.csv
./kaggle/input/gasherbrum-I.csv
./kaggle/input/annapurna-I.csv
./kaggle/input/makalu.csv


In [2]:
everest20_25 = pd.read_csv('./kaggle/input/everest_20-25.csv')

In [3]:
everest20_25

,Date,Name,Nationality,Age,Gender,Cause_of_Death,Altitude_meters,Location,Location_in_Meters,Route,Season,Weather_Conditions,Experience_Level,Experience_Level_Meters,Expedition_Company
0,1/5/2020,Li Yang,Czech Republic,56,Female,Altitude Sickness (AMS/HAPE/HACE),8400,Geneva Spur,NaN,East Face,Winter (Dec-Feb),Whiteout,Intermediate,1-2 previous 8000m peaks,Independent/Private
1,1/14/2020,Aiko Thomas,Finland,27,Female,Altitude Sickness (AMS/HAPE/HACE),8000,Death Zone,above 8000m,East Face,Winter (Dec-Feb),Snowstorm,Experienced,3-5 previous 8000m peaks,Asian Trekking
2,1/14/2020,Ingrid Torres,France,67,Male,Hypothermia,5364,Death Zone,above 8000m,Northeast Ridge Route (Tibet),Winter (Dec-Feb),Mixed conditions,Experienced,3-5 previous 8000m peaks,Himalayan Experience
3,1/18/2020,Wang Li,Brazil,36,Female,Falling,6065,Camp 3,7162m,Northeast Ridge Route (Tibet),Winter (Dec-Feb),High winds,Expert,6+ previous 8000m peaks,Summit Climb
4,1/25/2020,Chen Lewis,United States,53,Male,Avalanche,8750,South Summit,8750m,Southwest Face,Winter (Dec-Feb),High winds,Intermediate,1-2 previous 8000m peaks,Adventure Consultants
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,12/7/2025,Hiroshi Harris,South Korea,25,Female,Altitude Sickness (AMS/HAPE/HACE),8790,Khumbu Icefall,NaN,East Face,Winter (Dec-Feb),Extreme cold,Expert,6+ previous 8000m peaks,International Mountain Guides
496,12/15/2025,Steven Flores,China,30,Male,Cerebral edema,8000,Camp 3,7162m,Southwest Face,Winter (Dec-Feb),Extreme cold,Intermediate,1-2 previous 8000m peaks,Altitude Junkies
497,12/16/2025,Chen Liu,Finland,31,Female,Altitude Sickness (AMS/HAPE/HACE),8750,Western Cwm,NaN,North Face,Winter (Dec-Feb),Cloudy,Expert,6+ previous 8000m peaks,Asian Trekking
498,12/18/2025,Kenji Kumar,United Kingdom,24,Female,Rockfall,6400,Camp 3,7162m,North Face,Winter (Dec-Feb),High winds,Intermediate,1-2 previous 8000m peaks,Alpine Ascents


In [4]:
# Format dates
everest20_25['Date'] = pd.to_datetime(everest20_25['Date'], format='mixed', errors='coerce')
everest20_25['year'] = everest20_25['Date'].dt.year
everest20_25['month'] = everest20_25['Date'].dt.month

# Check if any dates failed to parse
print(f"Missing dates: {everest20_25['Date'].isna().sum()}")

# Deaths per year
everest20_25['year'].value_counts().sort_index()

Missing dates: 0


year
2020    86
2021    93
2022    92
2023    86
2024    73
2025    70
Name: count, dtype: int64

In [5]:
# Top 10 nationalities by death count
everest20_25['Nationality'].value_counts().head(10)

Nationality
Switzerland       31
South Africa      26
Russia            22
United States     22
South Korea       22
Brazil            21
Argentina         20
Norway            19
Denmark           19
Czech Republic    19
Name: count, dtype: int64

In [6]:
# Top 10 nationalities by death count
everest20_25['Nationality'].value_counts().tail(10)

Nationality
United Kingdom    14
India             13
Australia         13
Spain             13
Mexico            12
Nepal             12
Poland            10
Canada            10
Belgium            9
Israel             8
Name: count, dtype: int64

In [7]:
# Month when most deaths occur
everest20_25['month'] = pd.to_datetime(everest20_25['Date']).dt.month
everest20_25['month'].value_counts().sort_index()

month
1     44
2     45
3     33
4     39
5     52
6     26
7     54
8     34
9     46
10    41
11    47
12    39
Name: count, dtype: int64

In [8]:
# Basic overview
print("EVEREST 2020-2025 DATASET OVERVIEW")
print("=" * 60)
print(f"Total deaths: {len(everest20_25)}")
print(f"Date range: {everest20_25['Date'].min()} to {everest20_25['Date'].max()}")
print(f"Unique nationalities: {everest20_25['Nationality'].nunique()}")
print(f"Male/Female split:")
print(everest20_25['Gender'].value_counts())
print(f"\nAge statistics:")
print(everest20_25['Age'].describe())

EVEREST 2020-2025 DATASET OVERVIEW
Total deaths: 500
Date range: 2020-01-05 00:00:00 to 2025-12-29 00:00:00
Unique nationalities: 30
Male/Female split:
Gender
Male      252
Female    248
Name: count, dtype: int64

Age statistics:
count    500.000000
mean      36.278000
std       12.522233
min       20.000000
25%       28.000000
50%       31.000000
75%       43.000000
max       69.000000
Name: Age, dtype: float64


In [9]:
# Deaths by season
print("\nDeaths by Season:")
print(everest20_25['Season'].value_counts())


Deaths by Season:
Season
Summer (Jun-Aug)    193
Winter (Dec-Feb)    128
Spring (Apr-May)     91
Autumn (Oct-Nov)     88
Name: count, dtype: int64


In [10]:
# Deaths by cause
print("\nDeaths by Cause:")
print(everest20_25['Cause_of_Death'].value_counts())


Deaths by Cause:
Cause_of_Death
Altitude Sickness (AMS/HAPE/HACE)    129
Exhaustion                            81
Falling                               58
Hypothermia                           42
Avalanche                             36
Frostbite complications               27
Sudden cardiac event                  22
Cerebral edema                        20
Crevasse fall                         19
Icefall                               16
Rockfall                              15
Weather exposure                       9
Pulmonary edema                        9
Equipment failure                      9
Disorientation/Lost                    8
Name: count, dtype: int64


In [11]:
# Deaths by route
print("\nDeaths by Route:")
print(everest20_25['Route'].value_counts())


Deaths by Route:
Route
East Face                        88
Southwest Face                   87
South Col Route (Nepal)          86
North Face                       81
Northeast Ridge Route (Tibet)    79
West Ridge                       79
Name: count, dtype: int64


In [12]:
# Deaths by experience level
print("\nDeaths by Experience Level:")
print(everest20_25['Experience_Level'].value_counts())


Deaths by Experience Level:
Experience_Level
Novice                94
Experienced           86
Intermediate          85
Expert                85
Sherpa                76
Professional Guide    74
Name: count, dtype: int64


In [13]:
# Deaths by weather conditions
print("\nDeaths by Weather Conditions:")
print(everest20_25['Weather_Conditions'].value_counts())


Deaths by Weather Conditions:
Weather_Conditions
Extreme cold        75
Mixed conditions    66
High winds          63
Cloudy              63
Blizzard            60
Clear               60
Snowstorm           57
Whiteout            56
Name: count, dtype: int64


In [14]:
# Deaths by location on mountain
print("\nDeaths by Location:")
print(everest20_25['Location'].value_counts())


Deaths by Location:
Location
Khumbu Icefall    49
Camp 4            41
Western Cwm       41
Camp 2            38
South Summit      36
Balcony           35
Yellow Band       35
Camp 1            33
Summit            32
Death Zone        31
Lhotse Face       30
Camp 3            27
Hillary Step      27
Base Camp         23
Geneva Spur       22
Name: count, dtype: int64


In [15]:
# Age distribution by gender
print("\nAge by Gender:")
print(everest20_25.groupby('Gender')['Age'].describe())


Age by Gender:
        count       mean        std   min   25%   50%   75%   max
Gender                                                           
Female  248.0  36.439516  12.259524  20.0  28.0  31.0  42.0  69.0
Male    252.0  36.119048  12.797913  20.0  27.0  31.0  45.0  68.0


In [16]:
# Average age by cause of death (Female 36.34 years, Male 36.12 years)
print("\nAverage Age by Cause of Death:")
print(everest20_25.groupby('Cause_of_Death')['Age'].mean().sort_values(ascending=False))


Average Age by Cause of Death:
Cause_of_Death
Pulmonary edema                      39.666667
Hypothermia                          39.142857
Exhaustion                           37.555556
Avalanche                            37.444444
Falling                              37.051724
Disorientation/Lost                  36.750000
Frostbite complications              36.703704
Cerebral edema                       36.550000
Crevasse fall                        36.210526
Altitude Sickness (AMS/HAPE/HACE)    35.201550
Sudden cardiac event                 35.090909
Rockfall                             34.733333
Weather exposure                     34.555556
Icefall                              30.062500
Equipment failure                    29.888889
Name: Age, dtype: float64


In [17]:
# Age groups
everest20_25['age_group'] = pd.cut(everest20_25['Age'], 
                                    bins=[0, 30, 40, 50, 60, 100],
                                    labels=['Under 30', '30-40', '40-50', '50-60', '60+'])
print("\nDeaths by Age Group:")
print(everest20_25['age_group'].value_counts().sort_index())


Deaths by Age Group:
age_group
Under 30    218
30-40       145
40-50        45
50-60        50
60+          42
Name: count, dtype: int64


In [18]:
# Deaths by altitude zone
def categorize_altitude(alt):
    if pd.isna(alt):
        return 'Unknown'
    elif alt < 7000:
        return 'Below 7000m'
    elif alt < 8000:
        return '7000-8000m'
    elif alt < 8500:
        return '8000-8500m (Death Zone)'
    else:
        return 'Above 8500m (Summit Zone)'

everest20_25['altitude_zone'] = everest20_25['Altitude_meters'].apply(categorize_altitude)
print("\nDeaths by Altitude Zone:")
print(everest20_25['altitude_zone'].value_counts())

# Cause of death by altitude zone
print("\nCause of Death by Altitude Zone:")
print(pd.crosstab(everest20_25['altitude_zone'], everest20_25['Cause_of_Death']))


Deaths by Altitude Zone:
altitude_zone
8000-8500m (Death Zone)      209
Above 8500m (Summit Zone)    135
Below 7000m                  100
7000-8000m                    56
Name: count, dtype: int64

Cause of Death by Altitude Zone:
Cause_of_Death             Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
altitude_zone                                                             
7000-8000m                                                 5          9   
8000-8500m (Death Zone)                                   65         12   
Above 8500m (Summit Zone)                                 47          7   
Below 7000m                                               12          8   

Cause_of_Death             Cerebral edema  Crevasse fall  Disorientation/Lost  \
altitude_zone                                                                   
7000-8000m                              1              3                    1   
8000-8500m (Death Zone)                 8              4                  

In [19]:
# Gender vs Cause of Death
print("\nGender vs Cause of Death:")
print(pd.crosstab(everest20_25['Gender'], everest20_25['Cause_of_Death']))


Gender vs Cause of Death:
Cause_of_Death  Altitude Sickness (AMS/HAPE/HACE)  Avalanche  Cerebral edema  \
Gender                                                                         
Female                                         71         21              10   
Male                                           58         15              10   

Cause_of_Death  Crevasse fall  Disorientation/Lost  Equipment failure  \
Gender                                                                  
Female                      9                    2                  5   
Male                       10                    6                  4   

Cause_of_Death  Exhaustion  Falling  Frostbite complications  Hypothermia  \
Gender                                                                      
Female                  44       25                       12           17   
Male                    37       33                       15           25   

Cause_of_Death  Icefall  Pulmonary edema  Rockfall

In [20]:
# Experience Level vs Cause of Death
print("\nExperience Level vs Cause of Death:")
print(pd.crosstab(everest20_25['Experience_Level'], everest20_25['Cause_of_Death']))


Experience Level vs Cause of Death:
Cause_of_Death      Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
Experience_Level                                                   
Experienced                                        20          5   
Expert                                             25          5   
Intermediate                                       23          8   
Novice                                             26          5   
Professional Guide                                 15          6   
Sherpa                                             20          7   

Cause_of_Death      Cerebral edema  Crevasse fall  Disorientation/Lost  \
Experience_Level                                                         
Experienced                      3              4                    1   
Expert                           7              4                    0   
Intermediate                     3              1                    3   
Novice                           2              

In [21]:
# Season vs Cause of Death
print("\nSeason vs Cause of Death:")
print(pd.crosstab(everest20_25['Season'], everest20_25['Cause_of_Death']))


Season vs Cause of Death:
Cause_of_Death    Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
Season                                                           
Autumn (Oct-Nov)                                 18          9   
Spring (Apr-May)                                 25          8   
Summer (Jun-Aug)                                 56          9   
Winter (Dec-Feb)                                 30         10   

Cause_of_Death    Cerebral edema  Crevasse fall  Disorientation/Lost  \
Season                                                                 
Autumn (Oct-Nov)               8              2                    0   
Spring (Apr-May)               4              4                    1   
Summer (Jun-Aug)               6              8                    4   
Winter (Dec-Feb)               2              5                    3   

Cause_of_Death    Equipment failure  Exhaustion  Falling  \
Season                                                     
Autumn (Oct-Nov)       

In [22]:
# Weather vs Cause of Death
print("\nWeather Conditions vs Cause of Death:")
print(pd.crosstab(everest20_25['Weather_Conditions'], everest20_25['Cause_of_Death']))


Weather Conditions vs Cause of Death:
Cause_of_Death      Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
Weather_Conditions                                                 
Blizzard                                           15          4   
Clear                                              10          3   
Cloudy                                             21          2   
Extreme cold                                       19          5   
High winds                                         16          9   
Mixed conditions                                   20          6   
Snowstorm                                          12          4   
Whiteout                                           16          3   

Cause_of_Death      Cerebral edema  Crevasse fall  Disorientation/Lost  \
Weather_Conditions                                                       
Blizzard                         1              1                    1   
Clear                            2              3         

In [23]:
# Route vs Cause of Death
print("\nRoute vs Cause of Death:")
print(pd.crosstab(everest20_25['Route'], everest20_25['Cause_of_Death']))


Route vs Cause of Death:
Cause_of_Death                 Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
Route                                                                         
East Face                                                     22          4   
North Face                                                    22          4   
Northeast Ridge Route (Tibet)                                 23          6   
South Col Route (Nepal)                                       18          9   
Southwest Face                                                24          9   
West Ridge                                                    20          4   

Cause_of_Death                 Cerebral edema  Crevasse fall  \
Route                                                          
East Face                                   1              5   
North Face                                  2              5   
Northeast Ridge Route (Tibet)               3              0   
South Col Route (Nepa

In [24]:
# Independent vs Expedition Company
print("\nExpedition Type:")
everest20_25['is_independent'] = everest20_25['Expedition_Company'].str.contains('Independent', na=False)
print(everest20_25['is_independent'].value_counts())


Expedition Type:
is_independent
False    478
True      22
Name: count, dtype: int64


In [27]:
# Chi-square test: Is experience level independent of cause of death?
contingency = pd.crosstab(everest20_25['Experience_Level'], everest20_25['Cause_of_Death'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nExperience Level vs Cause of Death (Chi-square test):")
print(f"Chi-square: {chi2:.3f}, P-value: {p_value:.3f}")
if p_value < 0.05:
    print("Experience level and cause of death are related")
else:
    print("No significant relationship")


Experience Level vs Cause of Death (Chi-square test):
Chi-square: 60.043, P-value: 0.796
No significant relationship


In [28]:
# Correlation between numerical variables
numerical_cols = ['Age', 'Altitude_meters']
correlations = everest20_25[numerical_cols].corr()
print("\nCorrelation between Age and Altitude:")
print(correlations)


Correlation between Age and Altitude:
                      Age  Altitude_meters
Age              1.000000         0.009811
Altitude_meters  0.009811         1.000000


In [31]:
# EXPEDITION COMPANY DEEP DIVE
print("=" * 80)
print("EXPEDITION COMPANY ANALYSIS - EVEREST 2020-2025")
print("=" * 80)

EXPEDITION COMPANY ANALYSIS - EVEREST 2020-2025


In [33]:
# 1. Basic breakdown
print("\n1. EXPEDITION TYPE BREAKDOWN")
print("-" * 80)
everest20_25['expedition_type'] = everest20_25['Expedition_Company'].apply(
    lambda x: 'Independent/Private' if pd.isna(x) or 'Independent' in str(x) or 'Private' in str(x) 
    else 'Commercial Company'
)
print(everest20_25['expedition_type'].value_counts())
print(f"\nPercentage independent: {(everest20_25['expedition_type'] == 'Independent/Private').mean()*100:.1f}%")


1. EXPEDITION TYPE BREAKDOWN
--------------------------------------------------------------------------------
expedition_type
Commercial Company     478
Independent/Private     22
Name: count, dtype: int64

Percentage independent: 4.4%


In [34]:
# 2. Deaths by specific companies
print("\n2. DEATHS BY EXPEDITION COMPANY (Top 10)")
print("-" * 80)
company_deaths = everest20_25['Expedition_Company'].value_counts().head(10)
print(company_deaths)


2. DEATHS BY EXPEDITION COMPANY (Top 10)
--------------------------------------------------------------------------------
Expedition_Company
Climbing the Seven Summits       39
International Mountain Guides    38
Furtenbach Adventures            38
Peak Freaks                      37
Mountain Madness                 36
Summit Climb                     33
Adventure Consultants            33
Altitude Junkies                 33
Alpine Ascents                   32
Jagged Globe                     30
Name: count, dtype: int64


In [35]:
# 3. Independent vs Commercial: Key Metrics
print("\n3. INDEPENDENT VS COMMERCIAL COMPARISON")
print("-" * 80)
comparison = everest20_25.groupby('expedition_type').agg({
    'Age': ['mean', 'median', 'std'],
    'Altitude_meters': ['mean', 'median'],
    'Gender': lambda x: (x == 'Male').sum() / len(x) * 100  # % male
}).round(2)
print(comparison)


3. INDEPENDENT VS COMMERCIAL COMPARISON
--------------------------------------------------------------------------------
                       Age               Altitude_meters           Gender
                      mean median    std            mean  median <lambda>
expedition_type                                                          
Commercial Company   36.26   31.0  12.47         7795.97  8000.0    50.21
Independent/Private  36.64   30.5  13.86         7639.59  8000.0    54.55


In [37]:
# 4. Cause of Death by Expedition Type
print("\n4. CAUSE OF DEATH: INDEPENDENT VS COMMERCIAL")
print("-" * 80)
cause_by_type = pd.crosstab(everest20_25['expedition_type'], 
                             everest20_25['Cause_of_Death'], 
                             normalize='index') * 100
print(cause_by_type.round(1))


4. CAUSE OF DEATH: INDEPENDENT VS COMMERCIAL
--------------------------------------------------------------------------------
Cause_of_Death       Altitude Sickness (AMS/HAPE/HACE)  Avalanche  \
expedition_type                                                     
Commercial Company                                25.9        7.5   
Independent/Private                               22.7        0.0   

Cause_of_Death       Cerebral edema  Crevasse fall  Disorientation/Lost  \
expedition_type                                                           
Commercial Company              4.2            3.8                  1.5   
Independent/Private             0.0            4.5                  4.5   

Cause_of_Death       Equipment failure  Exhaustion  Falling  \
expedition_type                                               
Commercial Company                 1.9        15.5     11.7   
Independent/Private                0.0        31.8      9.1   

Cause_of_Death       Frostbite complicatio

In [38]:
# 5. Experience Level by Expedition Type
print("\n5. EXPERIENCE LEVEL: INDEPENDENT VS COMMERCIAL")
print("-" * 80)
exp_by_type = pd.crosstab(everest20_25['expedition_type'], 
                          everest20_25['Experience_Level'],
                          normalize='index') * 100
print(exp_by_type.round(1))



5. EXPERIENCE LEVEL: INDEPENDENT VS COMMERCIAL
--------------------------------------------------------------------------------
Experience_Level     Experienced  Expert  Intermediate  Novice  \
expedition_type                                                  
Commercial Company          16.9    17.2          17.2    18.4   
Independent/Private         22.7    13.6          13.6    27.3   

Experience_Level     Professional Guide  Sherpa  
expedition_type                                  
Commercial Company                 15.1    15.3  
Independent/Private                 9.1    13.6  


In [39]:
# 6. Route Choice by Expedition Type
print("\n6. ROUTE CHOICE: INDEPENDENT VS COMMERCIAL")
print("-" * 80)
route_by_type = pd.crosstab(everest20_25['expedition_type'], 
                            everest20_25['Route'],
                            normalize='index') * 100
print(route_by_type.round(1))


6. ROUTE CHOICE: INDEPENDENT VS COMMERCIAL
--------------------------------------------------------------------------------
Route                East Face  North Face  Northeast Ridge Route (Tibet)  \
expedition_type                                                             
Commercial Company        17.4        16.3                           16.3   
Independent/Private       22.7        13.6                            4.5   

Route                South Col Route (Nepal)  Southwest Face  West Ridge  
expedition_type                                                           
Commercial Company                      16.9            16.9        16.1  
Independent/Private                     22.7            27.3         9.1  


In [40]:
# 7. Season by Expedition Type
print("\n7. SEASON: INDEPENDENT VS COMMERCIAL")
print("-" * 80)
season_by_type = pd.crosstab(everest20_25['expedition_type'], 
                             everest20_25['Season'],
                             normalize='index') * 100
print(season_by_type.round(1))


7. SEASON: INDEPENDENT VS COMMERCIAL
--------------------------------------------------------------------------------
Season               Autumn (Oct-Nov)  Spring (Apr-May)  Summer (Jun-Aug)  \
expedition_type                                                             
Commercial Company               18.2              18.4              37.4   
Independent/Private               4.5              13.6              63.6   

Season               Winter (Dec-Feb)  
expedition_type                        
Commercial Company               25.9  
Independent/Private              18.2  


In [41]:
# 8. Weather Conditions by Expedition Type
print("\n8. WEATHER CONDITIONS: INDEPENDENT VS COMMERCIAL")
print("-" * 80)
weather_by_type = pd.crosstab(everest20_25['expedition_type'], 
                              everest20_25['Weather_Conditions'],
                              normalize='index') * 100
print(weather_by_type.round(1))


8. WEATHER CONDITIONS: INDEPENDENT VS COMMERCIAL
--------------------------------------------------------------------------------
Weather_Conditions   Blizzard  Clear  Cloudy  Extreme cold  High winds  \
expedition_type                                                          
Commercial Company       11.5   11.5    12.3          15.3        13.0   
Independent/Private      22.7   22.7    18.2           9.1         4.5   

Weather_Conditions   Mixed conditions  Snowstorm  Whiteout  
expedition_type                                             
Commercial Company               13.6       11.5      11.3  
Independent/Private               4.5        9.1       9.1  


In [43]:
# 10. Statistical Test: Are independent climbers dying at different altitudes?
from scipy import stats

independent_alt = everest20_25[everest20_25['expedition_type'] == 'Independent/Private']['Altitude_meters'].dropna()
commercial_alt = everest20_25[everest20_25['expedition_type'] == 'Commercial Company']['Altitude_meters'].dropna()

if len(independent_alt) > 0 and len(commercial_alt) > 0:
    t_stat, p_value = stats.ttest_ind(independent_alt, commercial_alt)
    print(f"\n10. STATISTICAL TEST: ALTITUDE DIFFERENCE")
    print("-" * 80)
    print(f"Independent avg altitude: {independent_alt.mean():.0f}m")
    print(f"Commercial avg altitude: {commercial_alt.mean():.0f}m")
    print(f"T-statistic: {t_stat:.3f}")
    print(f"P-value: {p_value:.3f}")
    if p_value < 0.05:
        print("✓ Statistically significant difference in death altitude")
    else:
        print("✗ No significant difference in death altitude")


10. STATISTICAL TEST: ALTITUDE DIFFERENCE
--------------------------------------------------------------------------------
Independent avg altitude: 7640m
Commercial avg altitude: 7796m
T-statistic: -0.707
P-value: 0.480
✗ No significant difference in death altitude


In [44]:
# 11. Top Commercial Companies - Detailed Analysis
print("\n11. TOP COMMERCIAL COMPANIES DETAILED ANALYSIS")
print("-" * 80)
commercial_only = everest20_25[everest20_25['expedition_type'] == 'Commercial Company']
top_companies = commercial_only['Expedition_Company'].value_counts().head(5).index

for company in top_companies:
    company_data = commercial_only[commercial_only['Expedition_Company'] == company]
    print(f"\n{company}:")
    print(f"  Total deaths: {len(company_data)}")
    print(f"  Avg age: {company_data['Age'].mean():.1f}")
    print(f"  Most common cause: {company_data['Cause_of_Death'].mode()[0] if len(company_data['Cause_of_Death'].mode()) > 0 else 'N/A'}")
    print(f"  Avg altitude: {company_data['Altitude_meters'].mean():.0f}m")


11. TOP COMMERCIAL COMPANIES DETAILED ANALYSIS
--------------------------------------------------------------------------------

Climbing the Seven Summits:
  Total deaths: 39
  Avg age: 36.5
  Most common cause: Altitude Sickness (AMS/HAPE/HACE)
  Avg altitude: 8009m

International Mountain Guides:
  Total deaths: 38
  Avg age: 34.3
  Most common cause: Altitude Sickness (AMS/HAPE/HACE)
  Avg altitude: 7935m

Furtenbach Adventures:
  Total deaths: 38
  Avg age: 40.7
  Most common cause: Altitude Sickness (AMS/HAPE/HACE)
  Avg altitude: 7781m

Peak Freaks:
  Total deaths: 37
  Avg age: 34.4
  Most common cause: Exhaustion
  Avg altitude: 7791m

Mountain Madness:
  Total deaths: 36
  Avg age: 35.6
  Most common cause: Exhaustion
  Avg altitude: 7512m
